In [ ]:
#!pip install datasets librosa soundfile huggingsound
#!pip install --upgrade datasets -> datasets update if there's an error

# An error related to the size of the files in dataset
"""
ValueError: The model corresponding to this feature extractor: Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}
 was trained using a sampling rate of 16000. Please make sure that the provided `raw_speech` input was sampled with 16000 and not 48000.
"""

# To solve:
#!pip install torchaudio

In [ ]:
# Does not work no more, because it's outdated
#!pip install huggingsound
#from huggingsound import SpeechRecognitionModel
"""
from huggingsound import SpeechRecognitionModel

model = SpeechRecognitionModel("jonatasgrosman/wav2vec2-large-xlsr-53-greek")
audio_paths = ["/path/to/file.mp3", "/path/to/another_file.wav"]

transcriptions = model.transcribe(audio_paths)
"""

In [ ]:
"""
updated
old one is:

from transformers import AutoProcessor, AutoModelForPreTraining

processor = AutoProcessor.from_pretrained("facebook/wav2vec2-large-xlsr-53")
model = AutoModelForPreTraining.from_pretrained("facebook/wav2vec2-large-xlsr-53")

"""
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

import torch
import soundfile as sf

In [ ]:
## import models
models = ["facebook/wav2vec2-large-xlsr-53", "jonatasgrosman/wav2vec2-large-xlsr-53-greek"]
model_base = Wav2Vec2ForCTC.from_pretrained(models[0])

model_greek = Wav2Vec2ForCTC.from_pretrained(models[1])
model_greek_processor = Wav2Vec2Processor.from_pretrained(models[1])

"""
facebook/wav2vec2-large-xlsr-53 error:
TypeError: expected str, bytes or os.PathLike object, not NoneType -> processor = Wav2Vec2Processor.from_pretrained(models[0])
"""


In [ ]:
from datasets import load_dataset, Audio

# Greek dataset
dataset = load_dataset("mozilla-foundation/common_voice_17_0", "el", split="train")

# Squeezing to 16hz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# print(dataset[1])

In [ ]:
# function to communicate with a model, TODO

def gen(dataset, model, processor):
    sample = dataset[1]
    speech = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    input = processor(speech, sampling_rate=sr, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(**input).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription

gen(dataset, model_greek, model_greek_processor)